# Kaggle remote benchmark

This thin notebook runs one repository benchmark configuration. Request preparation, prompts, image ordering, retry, parsing, resume, result writing, and metrics remain in repository modules and scripts. A fresh **Restart Session + Run All** is offline by default: it validates one sample and never reads Kaggle Secrets or calls the model API.

The dataset may be a released SciVer, SciAtomicBench, MuSciClaims, or SciClaimEval Task 1 file/directory, or an already-normalized JSON file. Registered adapters validate exact raw fields; they never infer mappings. Gold labels and source rationales are never sent to the model.

## 1. Configuration

Edit only this cell. Live modes require both the mode and the independent `RUN_LIVE_API=True` opt-in.

In [ ]:
GITHUB_REPOSITORY = "YOUR_ACCOUNT/SciVer"
GIT_REF = "feature/unified-benchmark-api"
EXPECTED_GIT_SHA = ""
DATASET_NAME = "SciVer"
DATASET_PATH = "/kaggle/input/YOUR_DATASET/testset.json"
MODEL_NAME = "Qwen2.5-VL-7B-Instruct"
METHOD = "cot"
EXPERIMENT_ID = "benchmark-v1"
RUN_MODE = "dry_run"
RUN_LIVE_API = False
PILOT_SAMPLE_COUNT = 20
REQUEST_DELAY = 1.0
OUTPUT_ROOT = "/kaggle/working/results"

## 2. Validate configuration and checkout

The repository is cloned only on a fresh session. Re-running never pulls or silently changes the checked-out commit. Set `EXPECTED_GIT_SHA` to pin an exact revision when required.

In [ ]:
from pathlib import Path
from urllib.parse import urlsplit
import html
import json
import os
import re
import shlex
import subprocess
import sys

VALID_RUN_MODES = {"dry_run", "smoke", "pilot", "full"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"RUN_MODE must be one of {sorted(VALID_RUN_MODES)}.")
if not isinstance(RUN_LIVE_API, bool):
    raise TypeError("RUN_LIVE_API must be exactly True or False.")
if RUN_MODE == "dry_run" and RUN_LIVE_API is not False:
    raise ValueError("dry_run requires RUN_LIVE_API=False.")
if RUN_MODE != "dry_run" and RUN_LIVE_API is not True:
    raise ValueError("Live modes require the independent RUN_LIVE_API=True opt-in.")
if isinstance(PILOT_SAMPLE_COUNT, bool) or not isinstance(PILOT_SAMPLE_COUNT, int) or PILOT_SAMPLE_COUNT < 1:
    raise ValueError("PILOT_SAMPLE_COUNT must be a positive integer.")
if isinstance(REQUEST_DELAY, bool) or not isinstance(REQUEST_DELAY, (int, float)) or REQUEST_DELAY < 0:
    raise ValueError("REQUEST_DELAY must be non-negative.")
if not GIT_REF and not EXPECTED_GIT_SHA:
    raise ValueError("Configure GIT_REF or EXPECTED_GIT_SHA.")
if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", GITHUB_REPOSITORY):
    raise ValueError("GITHUB_REPOSITORY must use the owner/repository form.")
if EXPECTED_GIT_SHA and not re.fullmatch(r"[0-9a-fA-F]{40}", EXPECTED_GIT_SHA):
    raise ValueError("EXPECTED_GIT_SHA must be empty or a full 40-character Git SHA.")
for name, value in {"DATASET_NAME": DATASET_NAME, "MODEL_NAME": MODEL_NAME, "METHOD": METHOD, "EXPERIMENT_ID": EXPERIMENT_ID}.items():
    if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9_.-]+", value):
        raise ValueError(f"{name} contains unsupported path characters.")

WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT_PATH = Path(OUTPUT_ROOT)
if not OUTPUT_ROOT_PATH.is_absolute() or WORKING_ROOT not in (OUTPUT_ROOT_PATH, *OUTPUT_ROOT_PATH.parents):
    raise ValueError("OUTPUT_ROOT must be an absolute path under /kaggle/working.")
REPOSITORY_DIR = WORKING_ROOT / GITHUB_REPOSITORY.rsplit("/", 1)[1]
RUN_OUTPUT_DIR = OUTPUT_ROOT_PATH / DATASET_NAME / MODEL_NAME / METHOD / EXPERIMENT_ID

def run_checked(command, *, cwd=None, env=None, check=True):
    safe_command = [str(part) for part in command]
    print("$", shlex.join(safe_command))
    return subprocess.run(safe_command, cwd=cwd, env=env, check=check)

repository_url = f"https://github.com/{GITHUB_REPOSITORY}.git"
if REPOSITORY_DIR.exists():
    if not (REPOSITORY_DIR / ".git").is_dir():
        raise RuntimeError(f"Existing checkout is not a Git repository: {REPOSITORY_DIR}")
else:
    clone_command = ["git", "clone"]
    if GIT_REF:
        clone_command.extend(["--branch", GIT_REF, "--single-branch"])
    clone_command.extend([repository_url, str(REPOSITORY_DIR)])
    run_checked(clone_command)

COMMIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR, text=True).strip()
if GIT_REF:
    configured_ref_sha = subprocess.check_output(["git", "rev-parse", f"{GIT_REF}^{{commit}}"], cwd=REPOSITORY_DIR, text=True).strip()
    if configured_ref_sha != COMMIT_SHA:
        raise RuntimeError("Existing checkout does not match GIT_REF. Start a fresh session or correct the configuration.")
if EXPECTED_GIT_SHA and COMMIT_SHA.casefold() != EXPECTED_GIT_SHA.casefold():
    raise RuntimeError("Checked-out commit does not match EXPECTED_GIT_SHA.")
print("Commit SHA:", COMMIT_SHA)

## 3. Install and verify offline

Only lightweight remote-path dependencies are installed; no model weights are downloaded. Tests run with credentials removed, model-download paths disabled, and the repository network guards active.

In [ ]:
run_checked(
    [
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "--no-cache-dir",
        "requests>=2.31.0,<3", "Pillow>=10,<12", "pytest>=8,<10",
    ],
    cwd=REPOSITORY_DIR,
)
offline_env = os.environ.copy()
offline_env.pop("API_KEY", None)
offline_env.pop("API_URL", None)
offline_env["CUDA_VISIBLE_DEVICES"] = ""
offline_env["HF_HUB_OFFLINE"] = "1"
offline_env["TRANSFORMERS_OFFLINE"] = "1"
run_checked([sys.executable, "-m", "compileall", "-q", "."], cwd=REPOSITORY_DIR, env=offline_env)
run_checked([sys.executable, "-m", "pytest", "-q"], cwd=REPOSITORY_DIR, env=offline_env)
run_checked([sys.executable, "main.py", "--help"], cwd=REPOSITORY_DIR, env=offline_env)

## 4. Dataset preflight and summary

Every record is passed through its strict released-schema adapter and the production request-preparation path before any secret is read. SciAtomic Markdown tables are rendered into local PNG evidence under the run directory. This validates normalized metadata, label mappings, reasoning method, context, captions, image existence/format, and ordered image count. Only aggregate values are displayed.

In [ ]:
if str(REPOSITORY_DIR) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIR))
from IPython.display import HTML, display
from utils.benchmark_workflow import preflight_dataset, require_adapter_ready

def display_records(records):
    if not records:
        display(HTML("<p>No rows.</p>"))
        return
    columns = list(records[0])
    header = "".join(f"<th>{html.escape(str(column))}</th>" for column in columns)
    rows = []
    for record in records:
        cells = []
        for column in columns:
            value = record.get(column)
            if isinstance(value, (dict, list)):
                value = json.dumps(value, sort_keys=True, ensure_ascii=False)
            cells.append(f"<td>{html.escape(str(value))}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML("<table><thead><tr>" + header + "</tr></thead><tbody>" + "".join(rows) + "</tbody></table>"))

DATASET_FILE = Path(DATASET_PATH)
selected_limit = {
    "dry_run": 1,
    "smoke": 1,
    "pilot": PILOT_SAMPLE_COUNT,
    "full": -1,
}[RUN_MODE]
DATASET_SUMMARY = preflight_dataset(
    DATASET_NAME, DATASET_FILE, selected_sample_count=selected_limit,
    evidence_dir=RUN_OUTPUT_DIR / "normalized_evidence" / DATASET_NAME,
)
summary_columns = {
    "dataset": DATASET_SUMMARY["dataset"],
    "split": DATASET_SUMMARY["split"],
    "total samples": DATASET_SUMMARY["total_samples"],
    "selected samples": DATASET_SUMMARY["selected_samples"],
    "label distribution": DATASET_SUMMARY["label_distribution"],
    "label support": DATASET_SUMMARY["label_support"],
    "reasoning or claim-type distribution": DATASET_SUMMARY["reasoning_or_claim_type_distribution"],
    "missing-context count": DATASET_SUMMARY["missing_context_count"],
    "missing-image count": DATASET_SUMMARY["missing_image_count"],
    "adapter readiness": DATASET_SUMMARY["adapter_readiness"],
}
display_records([summary_columns])
require_adapter_ready(DATASET_SUMMARY)

## 5. Live-only manifest and Secrets

Live modes share one deterministic directory: `/kaggle/working/results/<dataset>/<model>/<method>/<experiment_id>/`. A credential-free manifest is created before the first possible request. Resume is rejected if its immutable configuration fingerprint differs. Secrets are read only in live modes, and the endpoint must use HTTPS without embedded credentials.

In [ ]:
from utils.benchmark_workflow import build_run_manifest, ensure_run_manifest

RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if RUN_MODE != "dry_run":
    manifest = build_run_manifest(
        dataset_name=DATASET_NAME,
        dataset_path=DATASET_FILE,
        summary=DATASET_SUMMARY,
        model_name=MODEL_NAME,
        method=METHOD,
        experiment_id=EXPERIMENT_ID,
        request_delay=REQUEST_DELAY,
        git_sha=COMMIT_SHA,
    )
    manifest_status = ensure_run_manifest(RUN_OUTPUT_DIR / "run_manifest.json", manifest)
    print(f"Run manifest: {manifest_status}.")

    from kaggle_secrets import UserSecretsClient

    secret_client = UserSecretsClient()
    api_key_value = secret_client.get_secret("API_KEY")
    api_url_value = secret_client.get_secret("API_URL")
    if not isinstance(api_key_value, str) or not api_key_value.strip():
        raise RuntimeError("Kaggle Secret API_KEY is missing or empty.")
    if not isinstance(api_url_value, str) or not api_url_value.strip():
        raise RuntimeError("Kaggle Secret API_URL is missing or empty.")
    parsed_api_url = urlsplit(api_url_value)
    if parsed_api_url.scheme != "https" or not parsed_api_url.netloc or parsed_api_url.username is not None or parsed_api_url.password is not None:
        raise RuntimeError("Kaggle Secret API_URL must be an HTTPS URL without embedded credentials.")
    os.environ["API_KEY"] = api_key_value
    os.environ["API_URL"] = api_url_value
    del api_key_value, api_url_value, parsed_api_url, secret_client
    print("Live API configuration loaded from Kaggle Secrets; values are hidden.")
else:
    os.environ.pop("API_KEY", None)
    os.environ.pop("API_URL", None)
    print("dry_run: Kaggle Secrets were not accessed.")

## 6. Execute through `main.py`

`dry_run` validates one sample without a client. `smoke` sends exactly one sample. `pilot` uses `PILOT_SAMPLE_COUNT` with resume. `full` uses all samples (`--max-num -1`) with resume. No resume workflow uses overwrite.

In [ ]:
def benchmark_command(run_mode):
    max_num = {"dry_run": 1, "smoke": 1, "pilot": PILOT_SAMPLE_COUNT, "full": -1}[run_mode]
    command = [
        sys.executable, "main.py",
        "--provider", "remote",
        "--dataset", DATASET_NAME,
        "--dataset-path", str(DATASET_FILE),
        "--model", MODEL_NAME,
        "--method", METHOD,
        "--max-num", str(max_num),
        "--request-delay", str(REQUEST_DELAY),
        "--output-dir", str(RUN_OUTPUT_DIR),
    ]
    if run_mode == "dry_run":
        command.append("--dry-run")
    else:
        command.append("--live-api")
    if run_mode in {"pilot", "full"}:
        command.append("--resume")
    return command

run_command = benchmark_command(RUN_MODE)
if "--overwrite" in run_command:
    raise AssertionError("Live resume workflows must never use --overwrite.")
if RUN_MODE == "smoke" and run_command[run_command.index("--max-num") + 1] != "1":
    raise AssertionError("smoke must select exactly one sample.")
if RUN_MODE in {"pilot", "full"} and "--resume" not in run_command:
    raise AssertionError("pilot and full must use --resume.")
if RUN_MODE == "full" and run_command[run_command.index("--max-num") + 1] != "-1":
    raise AssertionError("full must use --max-num -1.")

run_env = os.environ.copy()
if RUN_MODE == "dry_run":
    run_env.pop("API_KEY", None)
    run_env.pop("API_URL", None)
completed_run = run_checked(run_command, cwd=REPOSITORY_DIR, env=run_env, check=False)
if completed_run.returncode not in ({0} if RUN_MODE == "dry_run" else {0, 1}):
    raise subprocess.CalledProcessError(completed_run.returncode, run_command)

## 7. Repository evaluation and result comparison

The offline evaluation script preserves SciVer Accuracy and reports per-dataset Accuracy, confusion matrix, parse coverage, and failures. When summaries for multiple datasets of the same model are evaluated together, it also reports unweighted macro-average Accuracy.

In [ ]:
result_files = sorted(RUN_OUTPUT_DIR.glob("*_*/*.jsonl"))
evaluation_datasets = []
macro_accuracy = []
if result_files:
    if len(result_files) != 1:
        raise RuntimeError("Expected exactly one result JSONL file in the deterministic run directory.")
    evaluation_dir = RUN_OUTPUT_DIR / "evaluation"
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    evaluation_json = evaluation_dir / "summary.json"
    evaluation_csv = evaluation_dir / "summary.csv"
    evaluation_command = [
        sys.executable, "scripts/evaluate_results.py", str(result_files[0]),
        "--summary-json", str(evaluation_json),
        "--csv", str(evaluation_csv),
    ]
    if DATASET_SUMMARY["label_support"] == "native_labels_preserved_unscored":
        evaluation_command.append("--allow-unscored-gold-labels")
    run_checked(evaluation_command, cwd=REPOSITORY_DIR, env=offline_env)
    evaluation_report = json.loads(evaluation_json.read_text(encoding="utf-8"))
    evaluation_datasets = evaluation_report["datasets"]
    macro_accuracy = evaluation_report["macro_average_accuracy"]

evaluated_samples = sum(group["total_samples"] for group in evaluation_datasets)
parsed_predictions = sum(group["successful_requests"] for group in evaluation_datasets)
api_failures = sum(group["api_failures"] for group in evaluation_datasets)
parse_failures = sum(group["parse_failures"] for group in evaluation_datasets)
invalid_inputs = sum(group["invalid_inputs"] for group in evaluation_datasets)
accuracy_numerator = sum(group["accuracy"]["numerator"] for group in evaluation_datasets)
accuracy_denominator = sum(group["accuracy"]["denominator"] for group in evaluation_datasets)
confusion_matrix = evaluation_datasets[0]["confusion_matrix"] if len(evaluation_datasets) == 1 else None
macro_average = macro_accuracy[0]["value"] if len(macro_accuracy) == 1 else None
comparison_row = {
    "dataset": DATASET_NAME,
    "model": MODEL_NAME,
    "method": METHOD,
    "selected samples": DATASET_SUMMARY["selected_samples"],
    "successful requests": parsed_predictions + parse_failures,
    "failed requests": api_failures,
    "invalid inputs": invalid_inputs,
    "parsed predictions": parsed_predictions,
    "parse coverage": parsed_predictions / evaluated_samples if evaluated_samples else None,
    "accuracy": accuracy_numerator / accuracy_denominator if accuracy_denominator else None,
    "confusion matrix": confusion_matrix,
    "macro-average Accuracy": macro_average,
    "output directory": str(RUN_OUTPUT_DIR),
}
display_records([comparison_row])